In [ ]:
# import os
# REPO_URL = 'https://github.com/tarakanov517/CV_JND_based_approach.git'   # <- впиши свой git
# BRANCH   = 'as/simkin_tests'

# %cd /kaggle/working
# if not os.path.exists('repo'):
#     !git clone -q --branch {BRANCH} {REPO_URL} repo
# %cd /kaggle/working/repo
# !pip -q install imagecorruptions joblib rootutils omegaconf datasets torchattacks
# import sys; sys.path.insert(0, os.getcwd())
# print('cwd:', os.getcwd())

In [ ]:
import os, time, math
from pathlib import Path
from statistics import NormalDist
 
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torchvision import datasets, transforms
 
SEED = 42
EPOCHS = 30                       # для первого прохода можно 30; «настоящий» ~100
BATCH_SIZE = 128
LR = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# JND-аугментация
EPSILON = 0.9
# WEBER_MAX = None
JND_SCALE = 1.0
L_BLACK, L_WHITE = 0.1, 300.0
GRID_SIZE = 64
# веса вспомогательных лоссов
# CONTRASTIVE_WEIGHT = 0.1
CONSISTENCY_WEIGHT = 1.0
AUGMIX_WEIGHT = 1.0
TEMPERATURE = 0.2
 
DATA_DIR = "/kaggle/input/datasets/pankrzysiu/cifar10-python"
CIFAR10C_DIR = "/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C" 
OUT_DIR = Path("cifar10c_jnd_results")
METHODS = ("jnd_consistency", "jnd_contrastive", "pixel_contrastive")

def seed_all(s):
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
 
 
seed_all(SEED)
print("device:", DEVICE, "| torch:", torch.__version__)

In [ ]:
class HistogramAdaptationModel:
    def __init__(self, exposure_time=1.0, angular_size_rad=1e-3, detection_probability=0.75,
                 absolute_threshold=1e-6, luminance_floor=1e-6):
        self.t, self.phi_d, self.p = float(exposure_time), float(angular_size_rad), float(detection_probability)
        self.L0, self.luminance_floor = float(absolute_threshold), float(luminance_floor)
 
    @staticmethod
    def _K(l, b1=0.98, b2=0.1, l1=2e-2):
        l = np.asarray(l, np.float64)
        return np.where(l <= 1.0, b1 * (1.0 + l1 / l), l ** b2)
 
    @staticmethod
    def _A(La, a1=4.8e6, a2=7.1e3, a3=4.5e-4):
        return 1.0 + np.sqrt(a1 * La) + a2 * La * (1.0 + np.sqrt(a3 * La))
 
    def _Lambda(self, l):
        return l * self._K(l)
 
    @staticmethod
    def _eta(La, L3=1e6, L4=1.0, a4=600.0):
        return 1.0 + L3 / (1.0 + a4 * La * (L4 + La))
 
    def _C(self, La, tau_min=0.05):
        return 1.0 + 1.0 / (self.t / (tau_min * self._eta(La)))
 
    @staticmethod
    def _phi1(La, c2=0.25, L1=10.0, L2=2.6e-5):
        return (1.0 + L1 / (L2 + La)) ** c2
 
    def _S(self, La, th1=6.1, ad=2.0):
        phi0 = self._phi1(La) * 0.5 * (1.0 / 60.0) * (np.pi / 180.0)
        return (th1 * phi0 / self.phi_d + 1.0) ** ad
 
    def _P(self):
        return abs(NormalDist().inv_cdf(self.p) / NormalDist().inv_cdf(0.75))
 
    def D(self, L, La):
        L = np.maximum(np.asarray(L, np.float64), self.luminance_floor)
        La = np.maximum(np.asarray(La, np.float64), self.luminance_floor)
        lam = L / (1e2 * self.L0 * self._A(La))
        return self.L0 * self._A(La) * self._C(La) * self._Lambda(lam) * self._S(La) * self._P()
 
    def fit(self, L, p, grid_size=GRID_SIZE):
        L, p = np.asarray(L, np.float64), np.asarray(p, np.float64)
        keep = p > 0
        L, p = np.maximum(L[keep], self.luminance_floor), p[keep] / p[keep].sum()
        cand = np.unique(np.r_[np.geomspace(L.min(), L.max(), grid_size), L])
        loss = np.array([np.sum(p * np.abs(np.log(self.D(L, c) / self.D(c, L)))) for c in cand])
        return float(cand[int(np.argmin(loss))])
 
 
threshold_model = HistogramAdaptationModel()
_LUMA = np.array([0.2126, 0.7152, 0.0722])
 
 
def _inv_srgb(c):
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
 
 
def _srgb(c):
    c = np.clip(c, 0, 1)
    return np.where(c <= 0.0031308, c * 12.92, 1.055 * c ** (1 / 2.4) - 0.055)
 
 
def estimate_La(img01):
    """img01: (H,W,3) [0,1] sRGB -> La из гистограммы яркости."""
    Y = (_inv_srgb(img01) * _LUMA).sum(-1)
    L = L_BLACK + (L_WHITE - L_BLACK) * Y
    counts, edges = np.histogram(L, bins=64, range=(L_BLACK, L_WHITE))
    centers = 0.5 * (edges[:-1] + edges[1:])
    return threshold_model.fit(centers, counts.astype(float))
 
 
# def make_jnd_view_rgb(x01, La, rng):
#     """x01: (B,H,W,3) [0,1] sRGB; La: (B,). Возмущаем только яркость, хрому сохраняем."""
#     lin = _inv_srgb(x01)
#     Y = (lin * _LUMA).sum(-1)
#     L = L_BLACK + (L_WHITE - L_BLACK) * Y
#     D = threshold_model.D(L, La[:, None, None])
#     D = np.minimum(D, WEBER_MAX * np.maximum(L, L_BLACK))
#     L2 = np.clip(L + rng.uniform(-EPSILON, EPSILON, L.shape) * D * JND_SCALE, L_BLACK, L_WHITE)
#     ratio = ((L2 - L_BLACK) / (L_WHITE - L_BLACK)) / np.maximum(Y, 1e-6)
#     return _srgb(np.clip(lin * ratio[..., None], 0, 1)).astype(np.float32)
 
 
# def make_pixel_view_rgb(x01, sigma, rng):
#     return np.clip(x01 + rng.normal(0, sigma, x01.shape), 0, 1).astype(np.float32)

In [ ]:
import math

_P_CONST = float(threshold_model._P())          # = 1.0 при p=0.75
_L0, _T, _PHI_D, _FLOOR = threshold_model.L0, threshold_model.t, threshold_model.phi_d, threshold_model.luminance_floor
_LUMA_T = torch.tensor([0.2126, 0.7152, 0.0722], device=DEVICE).view(1, 3, 1, 1)

def _A_t(La):   return 1.0 + torch.sqrt(4.8e6 * La) + 7.1e3 * La * (1.0 + torch.sqrt(4.5e-4 * La))
def _K_t(l):    return torch.where(l <= 1.0, 0.98 * (1.0 + 2e-2 / l), l.clamp_min(1e-12).pow(0.1))
def _eta_t(La): return 1.0 + 1e6 / (1.0 + 600.0 * La * (1.0 + La))
def _C_t(La):   return 1.0 + 1.0 / (_T / (0.05 * _eta_t(La)))
def _phi1_t(La):return (1.0 + 10.0 / (2.6e-5 + La)).pow(0.25)
def _S_t(La):
    phi0 = 0.5 * (1.0 / 60.0) * (math.pi / 180.0)
    return (6.1 * _phi1_t(La) * phi0 / _PHI_D + 1.0).pow(2.0)

def D_torch(L, La):
    L = L.clamp_min(_FLOOR); La = La.clamp_min(_FLOOR)
    A = _A_t(La)
    lam = L / (1e2 * _L0 * A)
    return _L0 * A * _C_t(La) * (lam * _K_t(lam)) * _S_t(La) * _P_CONST

def _inv_srgb_t(c): return torch.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055).pow(2.4))
def _srgb_t(c):
    c = c.clamp(0, 1)
    return torch.where(c <= 0.0031308, c * 12.92, 1.055 * c.pow(1 / 2.4) - 0.055)

def make_jnd_view_torch(x, La):
    lin = _inv_srgb_t(x)                                   # (B,3,H,W)
    Y = (lin * _LUMA_T).sum(1)                             # (B,H,W) линейная яркость
    L = L_BLACK + (L_WHITE - L_BLACK) * Y
    D = D_torch(L, La.view(-1, 1, 1))
    # if weber_max is not None:
    #     D = torch.minimum(D, weber_max * L.clamp_min(L_BLACK))
    noise = (torch.rand_like(L) * 2 - 1) * EPSILON         # U(-eps, eps)
    L2 = (L + noise * D * JND_SCALE).clamp(L_BLACK, L_WHITE)
    ratio = ((L2 - L_BLACK) / (L_WHITE - L_BLACK)) / Y.clamp_min(1e-6)
    return _srgb_t((lin * ratio.unsqueeze(1)).clamp(0, 1))



In [ ]:
class CIFARIndexed(datasets.CIFAR10):
    def __getitem__(self, i):
        img, y = super().__getitem__(i)
        return img, y, i
 
 
base_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),                       # [0,1], без нормализации (норм. внутри модели)
])
test_tf = transforms.ToTensor()
 
train_ds = CIFARIndexed(DATA_DIR, train=True,transform=base_tf)
test_ds = datasets.CIFAR10(DATA_DIR, train=False, transform=test_tf)
train_loader = torch.utils.data.DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_ds, 256, shuffle=False, num_workers=2)
 
# La на каждую train-картинку (по оригиналу, один раз)
_raw = datasets.CIFAR10(DATA_DIR, train=True).data.astype(np.float32) / 255.0   # (50000,32,32,3)
t0 = time.perf_counter()
La_train = np.empty(len(_raw), np.float32)
for i in range(len(_raw)):
    La_train[i] = estimate_La(_raw[i])
    if (i + 1) % 10000 == 0:
        print(f"  La {i+1}/{len(_raw)}")
print(f"La precompute: {time.perf_counter()-t0:.1f}s | La range [{La_train.min():.2f},{La_train.max():.2f}]")
 
# matched-RMS сигма пиксельного шума = RMS JND-вида
_r = np.random.default_rng(SEED)
La_train_t = torch.from_numpy(La_train).float().to(DEVICE)   # один раз

_sample = torch.from_numpy(_raw[:512]).permute(0, 3, 1, 2).to(DEVICE)
PIXEL_SIGMA = (make_jnd_view_torch(_sample, La_train_t[:512]) - _sample).std().item()
print("PIXEL_SIGMA:", round(PIXEL_SIGMA, 4))

def make_pixel_view_torch(x):
    return (x + torch.randn_like(x) * PIXEL_SIGMA).clamp(0, 1)

def to_views(imgs, indices, name):
    if name == "pixel_contrastive":
        return make_pixel_view_torch(imgs)
    return make_jnd_view_torch(imgs, La_train_t[indices.to(DEVICE)])   # weber из WEBER_MAX

def prepare_records(images, labels):
    """Предпосчёт на сплит: нормализованные пиксели, метки и La на картинку."""
    n = len(images)
    records = {
        "x": images.astype(np.float32).reshape(n, -1) / 255.0,
        "y": np.asarray(labels, dtype=np.int64),
        "La": np.empty(n),
    }
    t0 = time.perf_counter()
    for i, image in enumerate(images):
        records["La"][i] = estimate_La(image)
        if (i + 1) % 1000 == 0:
            print(f"  preprocessed {i + 1}/{n}")
    print(f"Preprocessed {n} images in {time.perf_counter() - t0:.1f}s")
    return records

# d = (make_jnd_view_torch(_sample, La_train_t[:512]).permute(0,2,3,1).cpu().numpy()
#      - make_jnd_view_rgb(_raw[:512], La_train[:512], np.random.default_rng(0)))
# print(f"Difference between numpy and torch: {d}")
# рандом разный, но диапазоны/статистика (rms, max) должны совпасть

In [ ]:
# from PIL import Image

# ids = np.arange(8)
# xt = torch.from_numpy(_raw[ids]).permute(0, 3, 1, 2).to(DEVICE)        # (8,3,32,32) [0,1]
# La_b = La_train_t[torch.from_numpy(ids).to(DEVICE)]

# aug = make_jnd_view_torch(xt, La_b).permute(0, 2, 3, 1).cpu().numpy()   # (8,32,32,3)
# orig = _raw[ids]                                                        # (8,32,32,3)

# # оригинал | возмущённый | разность×20 (по строке на картинку)
# rows = [np.concatenate([o, a, np.minimum(np.abs(o - a) * 20, 1)], axis=1) for o, a in zip(orig, aug)]
# canvas = np.concatenate(rows, axis=0)                                   # (8*32, 3*32, 3)
# display(Image.fromarray(np.uint8(canvas * 255)).resize((3 * 32 * 4, 8 * 32 * 4)))
# print("rms:", round((aug - orig).std(), 5), "| max:", round(np.abs(aug - orig).max(), 4))
# print("Labels:", _raw.__class__ and None)   # метки у тебя отдельно: train_ds.targets[ids] если нужно

In [ ]:
# import numpy as np, torch
# xb, La_b = _raw[:8], La_train[:8]

# # numpy: L и D как внутри make_jnd_view_rgb
# lin = _inv_srgb(xb); Y = (lin * _LUMA).sum(-1); L = L_BLACK + (L_WHITE - L_BLACK) * Y
# D_np = threshold_model.D(L, La_b[:, None, None])              # (8,32,32)

# # torch
# xt = torch.from_numpy(xb).permute(0, 3, 1, 2).to(DEVICE)
# lint = _inv_srgb_t(xt); Yt = (lint * _LUMA_T).sum(1); Lt = L_BLACK + (L_WHITE - L_BLACK) * Yt
# D_t = D_torch(Lt, La_train_t[:8].view(-1, 1, 1)).cpu().numpy()

# print("max|L_np-L_t| =", np.abs(L - Lt.cpu().numpy()).max())
# print("max|D_np-D_t| =", np.abs(D_np - D_t).max())
# print("rel err       =", (np.abs(D_np - D_t) / (np.abs(D_np) + 1e-9)).max())



# r = np.random.default_rng(0)
# v_np = make_jnd_view_rgb(xb, La_b, r)                               # numpy, (8,32,32,3)
# v_t  = make_jnd_view_torch(xt, La_train_t[:8]).permute(0, 2, 3, 1).cpu().numpy()
# d_np, d_t = v_np - xb, v_t - xb
# print("rms  np/t:", round(d_np.std(), 5), round(d_t.std(), 5))
# print("max  np/t:", round(np.abs(d_np).max(), 4), round(np.abs(d_t).max(), 4))
# print("mean np/t:", round(np.abs(d_np).mean(), 5), round(np.abs(d_t).mean(), 5))

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, inp, out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(inp, out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out)
        self.conv2 = nn.Conv2d(out, out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out)
        self.short = nn.Sequential()
        if stride != 1 or inp != out:
            self.short = nn.Sequential(nn.Conv2d(inp, out, 1, stride, bias=False), nn.BatchNorm2d(out))
 
    def forward(self, x):
        o = F.relu(self.bn1(self.conv1(x)))
        o = self.bn2(self.conv2(o))
        return F.relu(o + self.short(x))
 
 
class ResNet18(nn.Module):
    def __init__(self, num_classes=10, proj_dim=128):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make(64, 64, 2, 1)
        self.layer2 = self._make(64, 128, 2, 2)
        self.layer3 = self._make(128, 256, 2, 2)
        self.layer4 = self._make(256, 512, 2, 2)
        self.classifier = nn.Linear(512, num_classes)
        self.projector = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, proj_dim))
 
    def _make(self, inp, out, n, stride):
        return nn.Sequential(BasicBlock(inp, out, stride), *[BasicBlock(out, out) for _ in range(n - 1)])
 
    def forward(self, x):
        x = (x - self.mean) / self.std
        o = F.relu(self.bn1(self.conv1(x)))
        o = self.layer4(self.layer3(self.layer2(self.layer1(o))))
        o = F.adaptive_avg_pool2d(o, 1).flatten(1)
        return self.classifier(o), F.normalize(self.projector(o), dim=1)

In [ ]:
def nt_xent(z0, z1, labels, temperature=TEMPERATURE, class_aware=True):
    b = len(labels)
    z = torch.cat([z0, z1])
    scores = z @ z.T / temperature
    idx = torch.arange(2 * b, device=z.device)
    positive = (idx + b) % (2 * b)
    invalid = torch.eye(2 * b, dtype=torch.bool, device=z.device)
    if class_aware:
        y = labels.repeat(2)
        invalid = invalid | (y[:, None] == y[None, :])
        invalid[idx, positive] = False
    return F.cross_entropy(scores.masked_fill(invalid, float("-inf")), positive)
 
 
def jsd(*logits):
    logp = [F.log_softmax(l, 1) for l in logits]
    p = [lp.exp() for lp in logp]
    logm = torch.clamp(torch.stack(p).mean(0), 1e-8).log()
    return sum((pi * (lpi - logm)).sum(1).mean() for pi, lpi in zip(p, logp)) / len(p)
 
 
try:
    augmix_tf = transforms.AugMix(severity=3, mixture_width=3, chain_depth=-1, alpha=1.0, all_ops=False)                     # torchvision >= 0.13, работает с uint8
    def augmix_views(x01):
        xu = (x01 * 255).to(torch.uint8)
        return (augmix_tf(xu).float() / 255.0, augmix_tf(xu).float() / 255.0)
except Exception as _e:
    augmix_tf = None
    print("AugMix недоступен в этой версии torchvision:", _e)

In [ ]:
# def to_views(imgs, indices, name, rng):
#     """Вернуть второй вид (xp) для аугментированных методов, на GPU."""
#     x01 = imgs.permute(0, 2, 3, 1).cpu().numpy()                 # (B,H,W,3) [0,1]
#     if name == "pixel_contrastive":
#         xp = make_pixel_view_rgb(x01, PIXEL_SIGMA, rng)
#     else:
#         xp = make_jnd_view_rgb(x01, La_train[indices.numpy()], rng)
#     return torch.from_numpy(xp).permute(0, 3, 1, 2).to(imgs.device)
 
 
def train_method(name, epochs=EPOCHS):
    seed_all(SEED)
    net = ResNet18().to(DEVICE)
    opt = torch.optim.SGD(net.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4, nesterov=True)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for epoch in range(epochs):
        net.train()
        t0 = time.perf_counter()
        for imgs, y, idx in train_loader:
            imgs, y = imgs.to(DEVICE), y.to(DEVICE)
            if name == "clean_ce":
                logits, _ = net(imgs)
                loss = F.cross_entropy(logits, y)
            elif name == "augmix":
                a1, a2 = augmix_views(imgs)
                logits, _ = net(torch.cat([imgs, a1, a2]))
                lc, l1, l2 = logits.chunk(3)
                loss = F.cross_entropy(lc, y) + AUGMIX_WEIGHT * jsd(lc, l1, l2)

            else:
                xp = to_views(imgs, idx, name)
                logits, z = net(torch.cat([imgs, xp]))
                ce = F.cross_entropy(logits, torch.cat([y, y]))
                b = len(y)
                if name in ("jnd_contrastive", "pixel_contrastive"):
                    loss = ce + CONTRASTIVE_WEIGHT * nt_xent(z[:b], z[b:], y)
                elif name == "jnd_consistency":
                    loss = ce + CONSISTENCY_WEIGHT * jsd(logits[:b], logits[b:])
                else:                                           # jnd_ce
                    loss = ce
            opt.zero_grad()
            loss.backward()

            if name == 'augmix' or name == 'jnd_consistency':
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        sched.step()
        if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
            acc = evaluate(net, test_loader)
            print(f"{name} ep{epoch+1}/{epochs} clean={acc:.4f} ({time.perf_counter()-t0:.0f}s/ep)")
    return net
 
 
@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    c = t = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        c += (net(x)[0].argmax(1) == y).sum().item()
        t += y.size(0)
    return c / t
 
 
models = {}
for m in METHODS:
    print(f"\n===== {m} =====")
    if m in ('jnd_contrastive', 'pixel_contrastive'):
        for w in (0.3, 1.0, 3.0):
            CONTRASTIVE_WEIGHT = w 
            models[f"{m}_weight_{w}"] = train_method(m)
    else:
        models[f"{m}_weight_0"] = train_method(m)
 
# OUT_DIR.mkdir(exist_ok=True)
# for m, net in models.items():
#     torch.save(net.state_dict(), OUT_DIR / f"{m}.pt")

In [ ]:
GROUPS = {
    "noise": ["gaussian_noise", "shot_noise", "impulse_noise"],
    "blur": ["defocus_blur", "glass_blur", "motion_blur", "zoom_blur"],
    "weather": ["snow", "frost", "fog", "brightness"],
    "digital": ["contrast", "elastic_transform", "pixelate", "jpeg_compression"],
}
 
 
@torch.no_grad()
def acc_np(net, imgs_u8, labels):
    net.eval()
    c = t = 0
    for i in range(0, len(labels), 256):
        x = torch.from_numpy(imgs_u8[i:i + 256]).permute(0, 3, 1, 2).float().div(255).to(DEVICE)
        y = torch.from_numpy(labels[i:i + 256]).to(DEVICE)
        c += (net(x)[0].argmax(1) == y).sum().item()
        t += len(y)
    return c / t
 
 
cdir = Path(CIFAR10C_DIR)
labels_c = np.load(cdir / "labels.npy")[:10000]
rows = {}
for m, net in models.items():
    clean = evaluate(net, test_loader)
    per_group = {}
    for g, corrs in GROUPS.items():
        rets = []
        for c in corrs:
            f = cdir / f"{c}.npy"
            if not f.exists():
                continue
            arr = np.load(f)                                   # (50000,32,32,3), severity = блоки по 10000
            accs = [acc_np(net, arr[s * 10000:(s + 1) * 10000], labels_c) for s in range(5)]
            rets.append(np.mean(accs) / clean)
        per_group[g] = float(np.mean(rets)) if rets else float("nan")
    overall = float(np.nanmean(list(per_group.values())))
    rows[m] = dict(clean=clean, **per_group, avg=overall)
    print(f"{m:18s} clean={clean:.3f} | " +
          " ".join(f"{g}={per_group[g]:.3f}" for g in GROUPS) + f" | avg={overall:.3f}")
 
import json
(OUT_DIR / "cifar10c_retention.json").write_text(json.dumps(rows, indent=2))
print("\nSaved:", OUT_DIR / "cifar10c_retention.json")
 